# 1. Set up MLFlow

In [1]:
%pip install mlflow -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import mlflow

In [3]:
# Define the MLflow storage path in Google Drive
mlflow_storage_path = "/Users/dianaterraza/Desktop/corporacion_favorita/mlflow_results"

# Set MLflow to log to the directory
mlflow.set_tracking_uri(f"file:{mlflow_storage_path}")

In [4]:
# Set up experiment name
mlflow.set_experiment("Demand Forecast Experiment")

<Experiment: artifact_location='file:///Users/dianaterraza/Desktop/corporacion_favorita/mlflow_results/694996747944737169', creation_time=1736847837127, experiment_id='694996747944737169', last_update_time=1736847837127, lifecycle_stage='active', name='Demand Forecast Experiment', tags={}>

## Start MLFlow UI

In [5]:
with mlflow.start_run(run_name="example_run"):
    mlflow.log_param("param1", 5)
    mlflow.log_metric("metric1", 0.87)

In [6]:
%pip install pyngrok -q


Note: you may need to restart the kernel to use updated packages.


In [7]:
from pyngrok import ngrok, conf
import getpass
import subprocess

In [8]:
subprocess.Popen(["mlflow", "ui", "--backend-store-uri", mlflow_storage_path])

<Popen: returncode: None args: ['mlflow', 'ui', '--backend-store-uri', '/Use...>

[2025-01-14 11:20:05 +0100] [3439] [INFO] Starting gunicorn 23.0.0
[2025-01-14 11:20:05 +0100] [3439] [INFO] Listening at: http://127.0.0.1:5000 (3439)
[2025-01-14 11:20:05 +0100] [3439] [INFO] Using worker: sync
[2025-01-14 11:20:05 +0100] [3440] [INFO] Booting worker with pid: 3440
[2025-01-14 11:20:05 +0100] [3441] [INFO] Booting worker with pid: 3441
[2025-01-14 11:20:05 +0100] [3442] [INFO] Booting worker with pid: 3442
[2025-01-14 11:20:05 +0100] [3443] [INFO] Booting worker with pid: 3443


# Import all libraries we will need for the modeling and evaluation

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import joblib
import mlflow.pyfunc
from sklearn.metrics import mean_absolute_error, mean_squared_error

from darts import TimeSeries
from darts.models import ARIMA
from darts.metrics import mae, mape, rmse

import xgboost as xgb

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Read the files with pandas

In [10]:
# Load the CSV files into pandas DataFrames
df_stores = pd.read_csv('/Users/dianaterraza/Desktop/Data/stores.csv')
df_items = pd.read_csv('/Users/dianaterraza/Desktop/Data/items.csv')
df_transactions = pd.read_csv('/Users/dianaterraza/Desktop/Data/transactions.csv')
df_oil = pd.read_csv('/Users/dianaterraza/Desktop/Data/oil.csv')
df_holidays_events = pd.read_csv('/Users/dianaterraza/Desktop/Data/holidays_events.csv')
df_train = pd.read_csv('/Users/dianaterraza/Desktop/Data/train.csv')
df_test = pd.read_csv('/Users/dianaterraza/Desktop/Data/test.csv')

/var/folders/p6/1v6w0vgj3d951b4r8sgj0lgw0000gn/T/ipykernel_3418/2255240867.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('/Users/dianaterraza/Desktop/Data/train.csv')


## For the quick experimenting, we will select a few store-product pairs

In [43]:
# Let's filter the data for one store and one item to keep it simple
store_ids = [1]
item_ids = [96995,99197,2134058,103501,103520]
#Select data January-March 2014 period
start_date = '2014-01-01'
end_date = '2014-03-31'


# Initialize an empty list to hold filtered chunks
filtered_chunks = []

# Define the chunk size (number of rows per chunk)
chunk_size = 10 ** 6  # Adjust based on your system's memory capacity

# Read the CSV file in chunks
for chunk in pd.read_csv('/Users/dianaterraza/Desktop/Data/train.csv', chunksize=chunk_size):
    # Filter the chunk for the desired store IDs
    chunk_filtered = chunk[(chunk['store_nbr'].isin(store_ids)) &
                            (chunk['date'] >= start_date) &
                            (chunk['date'] <= end_date)]
    # Append the filtered chunk to the list
    filtered_chunks.append(chunk_filtered)
    # Optional: Delete the chunk to free up memory
    del chunk

# Concatenate all filtered chunks into a single DataFrame
df_filtered = pd.concat(filtered_chunks, ignore_index=True)

# Clean up to free memory
del filtered_chunks

/var/folders/p6/1v6w0vgj3d951b4r8sgj0lgw0000gn/T/ipykernel_3418/1761552812.py:16: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv('/Users/dianaterraza/Desktop/Data/train.csv', chunksize=chunk_size):


# Prepare data

## Fill out missing dates with 0 sales

In [42]:
import pandas as pd

# Convert 'date' column to datetime format
df_filtered['date'] = pd.to_datetime(df_filtered['date'])

# Get the minimum and maximum dates in the dataset to create a full date range
min_date = df_filtered['date'].min()
max_date = df_filtered['date'].max()
print(min_date.date(), max_date.date())

# Create a full date range covering all days between the min and max dates
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')

# Create an empty DataFrame to store the final result
df_filled = pd.DataFrame()

# Iterate through each store and item combination
for (store, item), group in df_filtered.groupby(['store_nbr', 'item_nbr']):
    # Set 'date' as index and sort by date
    group.set_index('date', inplace=True)
    group = group.sort_index()

    # Reindex to fill missing dates with 0 sales
    group = group.reindex(full_date_range, fill_value=0)

    # Keep track of the store and item number for each row
    group['store_nbr'] = store
    group['item_nbr'] = item

    # Ensure that missing sales values are filled with 0
    group['unit_sales'] = group['unit_sales'].fillna(0)

    # Append the group to the final DataFrame
    df_filled = pd.concat([df_filled, group])

# Reset the index to get 'date' back as a column
df_filled.reset_index(inplace=True)
df_filled.rename(columns={'index': 'date'}, inplace=True)

2014-01-02 2014-03-31


# Split in test and train datasets

In [44]:
split_date = '2014-01-01'
train = df_filled[df_filled['date'] < split_date]
test = df_filled[df_filled['date'] >= split_date]
print("Train dataframe shape:",train.shape)
print("Test dataframe shape:",test.shape)

Train dataframe shape: (0, 6)
Test dataframe shape: (199449, 6)


### ERROR!:
When i try to run the ARIMA model i will get an error because my entry Train dataframe (train_series) has no entries, it contains 0 elements and ARIMA(p=5) model requires at least 30 entries. The exercise provide the dates of sales in Guayas from January-March 2014 

To fix this error i could add a fallback or handlig small trainin data by changing the split date or using different strategy to split:

In [45]:
train_size = int(len(group) * 0.8)
train = group.iloc[:train_size]
test = group.iloc[train_size:]

In [46]:
if train_series.shape[0] < 30:
    print(f"Not enough data for item {item_nbr} and store {store_nbr}. Skipping this combination.")

Not enough data for item 96995 and store 1. Skipping this combination.


I might want to change the split date to a time when there is more data available for training. For instance, if the split date is too close to the beginning of my data, there might not be enough data to train the model.

# Forecast with ARIMA

### Loop Through Each Product-Store Pair and Apply ARIMA Separately
The ARIMA model in Darts is designed for univariate time series forecasting, meaning it works with a single time series at a time.
In the code below we fit the model for each store-item pair and log the results in MLFlow.

In [54]:
import os
from tempfile import TemporaryDirectory
import mlflow
from darts import TimeSeries
from darts.models import ARIMA
import matplotlib.pyplot as plt

split_date = '2013-12-01'  # Use an earlier date for the split

rmad_values = []
bias_values = []
rmse_values = []
plot_paths = []
plot_count = 0

with mlflow.start_run(run_name="arima_run"):
    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")
    # Log ARIMA parameters only once for the whole run
    mlflow.log_param("p", 5)
    mlflow.log_param("d", 1)
    mlflow.log_param("q", 0)

    for (item_nbr, store_nbr), group in df_filled.groupby(['item_nbr', 'store_nbr']):
        group = group.groupby(['date']).sum()['unit_sales'].reset_index()

        # Check data before splitting
        print(f"Processing item {item_nbr} at store {store_nbr}")
        print(f"Before {split_date}: {group[group['date'] < split_date].shape[0]} entries")
        print(f"After {split_date}: {group[group['date'] >= split_date].shape[0]} entries")

        # Create train and test series based on the split date
        train_group = group[group['date'] < split_date]
        test_group = group[group['date'] >= split_date]

        # Skip if not enough data for training
        if len(train_group) < 30:
            print(f"Not enough data for item {item_nbr} and store {store_nbr}. Skipping this combination.")
            continue

        # Create TimeSeries objects for train and test datasets
        train_series = TimeSeries.from_dataframe(train_group, value_cols='unit_sales', time_col='date', fill_missing_dates=True, freq='D')
        train_series += 1e-5  # Add a small constant to make all values positive
        test_series = TimeSeries.from_dataframe(test_group, value_cols='unit_sales', time_col='date', fill_missing_dates=True, freq='D')
        test_series += 1e-5  # Add a small constant to make all values positive

        # Initialize ARIMA model with (p, d, q) parameters
        arima_model = ARIMA(p=5, d=1, q=0)

        # Fit the ARIMA model on the training data
        arima_model.fit(train_series)

        # Forecast the next values (the same length as the test set)
        arima_forecast = arima_model.predict(len(test_series))

        # Save and log model as an artifact
        with TemporaryDirectory() as tmp_dir:
            model_path = os.path.join(tmp_dir, f"arima_model_store_{store_nbr}_item_{item_nbr}.pkl")
            arima_model.save(model_path)  # Save ARIMA model
            mlflow.log_artifact(model_path, artifact_path=f"models/arima_store_{store_nbr}_item_{item_nbr}")

        # Optional: plot forecast vs actual data for a subset
        if plot_count < 3:
            plt.figure(figsize=(12, 6))  # Adjust the figure size (width, height)
            train_series.plot(label='Training Data')
            test_series.plot(label='Test Data')
            arima_forecast.plot(label='ARIMA Forecast', color='red')
            plt.title(f'Daily Sales of Item {item_nbr} at Store {store_nbr}', fontsize=20, fontweight='bold')
            plt.xlabel('Date', fontsize=16)
            plt.ylabel('Unit Sales', fontsize=16)
            plt.xticks(fontsize=14, rotation=45)
            plt.yticks(fontsize=14)
            plt.legend()
            file_path = f'ARIMA_forecast_store_{store_nbr}_item_{item_nbr}.png'
            plt.savefig(file_path)  # Saves the plot as a PNG file
            plt.show()
            plot_paths.append(file_path)
            plot_count += 1

        # Calculate metrics
        bias_value = (arima_forecast.values() - test_series.values()).mean()
        bias_values.append(bias_value)
        rmse_value = rmse(test_series, arima_forecast)
        rmse_values.append(rmse_value)
        mae_value = mae(test_series, arima_forecast)
        mean_actual = test_series.values().mean()

        # Handle the division by zero case for rmad_value calculation
        rmad_value = mae_value / mean_actual if mean_actual != 0 else float('inf')  # Avoid division by zero
        rmad_values.append(rmad_value)

    # Log average metrics across all models
    mlflow.log_metric("Average_rMAD", sum(rmad_values) / len(rmad_values))
    mlflow.log_metric("Average_Bias", sum(bias_values) / len(bias_values))
    mlflow.log_metric("Average_RMSE", sum(rmse_values) / len(rmse_values))

    # Log plots
    for plot_path in plot_paths:
        mlflow.log_artifact(plot_path, artifact_path="plots")


Run ID: c3d11d3e7f9c4e9d964b5c906a13eb15
Processing item 96995 at store 1
Before 2013-12-01: 0 entries
After 2013-12-01: 89 entries
Not enough data for item 96995 and store 1. Skipping this combination.
Processing item 103520 at store 1
Before 2013-12-01: 0 entries
After 2013-12-01: 89 entries
Not enough data for item 103520 and store 1. Skipping this combination.
Processing item 103665 at store 1
Before 2013-12-01: 0 entries
After 2013-12-01: 89 entries
Not enough data for item 103665 and store 1. Skipping this combination.
Processing item 105574 at store 1
Before 2013-12-01: 0 entries
After 2013-12-01: 89 entries
Not enough data for item 105574 and store 1. Skipping this combination.
Processing item 105575 at store 1
Before 2013-12-01: 0 entries
After 2013-12-01: 89 entries
Not enough data for item 105575 and store 1. Skipping this combination.
Processing item 105577 at store 1
Before 2013-12-01: 0 entries
After 2013-12-01: 89 entries
Not enough data for item 105577 and store 1. Skip

ZeroDivisionError: division by zero